In [0]:
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/gold/order_items"

In [0]:
# -- # Read clean order_items from silver and dimensions from gold to make fact table order_items of gold
silver_order_items_df = (
    spark.readStream \
        .format("delta") \
        .option("readChangeData", "true") \
        .table("retails.silver.order_items_cleaned") \
        .select("order_item_id","order_item_order_id", "order_item_product_id", "order_item_quantity", "order_item_subtotal", "order_item_product_price")
)

silver_orders_df = (
    spark.read \
        .format("delta") \
        .table("retails.silver.orders_cleaned") \
        .select("order_id", "order_customer_id", "order_date")
)

dim_customers_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_customers") \
        .select("customer_key","customer_id")
)

dim_products_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_products") \
        .select("product_key","product_id", "product_category_id")
)

dim_categories_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_categories") \
        .select("category_key","category_id", "category_department_id")
)

dim_departments_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_departments") \
        .select("department_key", "department_id")
)

dim_dates_df = (
    spark.read \
        .format("delta") \
        .table("retails.gold.dim_dates") \
        .select("date_key", "full_date")
)


In [0]:
from pyspark.sql.functions import col, current_timestamp

fact_order_items_df = (
    silver_order_items_df \
        .join(silver_orders_df, silver_order_items_df.order_item_order_id == silver_orders_df.order_id, "inner") \
        .join(dim_dates_df, silver_orders_df.order_date == dim_dates_df.full_date, "inner") \
        .join(dim_customers_df, silver_orders_df.order_customer_id == dim_customers_df.customer_id, "inner") \
        .join(dim_products_df, silver_order_items_df.order_item_product_id == dim_products_df.product_id, "inner") \
        .join(dim_categories_df, dim_products_df.product_category_id == dim_categories_df.category_id, "inner") \
        .join(dim_departments_df, dim_categories_df.category_department_id == dim_departments_df.department_id, "inner") \
        
        .select("order_item_id", col("order_item_order_id").alias("order_id"), dim_customers_df.customer_key, dim_products_df.product_key, dim_categories_df.category_key, dim_departments_df.department_key, dim_dates_df.date_key.alias("order_date_key"), col("order_item_quantity").alias("quantity"), col("order_item_subtotal").alias("subtotal"), col("order_item_product_price").alias("product_price"))
)

fact_order_items_df = fact_order_items_df.withColumn("created_ts", current_timestamp()) \
                                        .withColumn("quantity", col("quantity").cast("int")) \
                                        .withColumn("order_date_key", col("order_date_key").cast("bigint")) \
                                        .withColumn("subtotal", col("subtotal").cast("double")) \
                                        .withColumn("product_price", col("product_price").cast("double"))


In [0]:
def upsert_fact_order_items(batch_df, batch_id):
    batch_df.write \
        .mode("append") \
        .format("delta") \
        .saveAsTable("retails.gold.fact_order_items")

In [0]:
fact_order_items_df.writeStream \
                    .outputMode("append") \
                    .foreachBatch(upsert_fact_order_items) \
                    .option("checkpointLocation", _checkpoints) \
                    .trigger(availableNow=True) \
                    .start() \
                    .awaitTermination()
                    

In [0]:
%sql
select count(*) from retails.gold.fact_order_items;

-- truncate table retails.gold.fact_order_items;

In [0]:
# %sql
# select order_item_id, order_item_order_id, dc.customer_key, dp.product_key, dcat.category_key, ddep.department_key, dd.date_key, order_item_quantity, order_item_subtotal, order_item_product_price
# from retails.silver.order_items_cleaned oic

#     inner join retails.silver.orders_cleaned oc
#         on oic.order_item_order_id = oc.order_id
    
#     inner join retails.gold.dim_dates dd
#         on oc.order_date = dd.full_date
        
#     inner join retails.gold.dim_customers dc
#         on oc.order_customer_id = dc.customer_id

#     inner join retails.gold.dim_products dp
#         on oic.order_item_product_id = dp.product_id
    
#     inner join retails.gold.dim_categories dcat
#         on dp.product_category_id = dcat.category_id
    
#     inner join retails.gold.dim_departments ddep
#         on dcat.category_department_id = ddep.department_id

# order by order_item_order_id

In [0]:
# dbutils.fs.ls(_checkpoints)
# dbutils.fs.rm(_checkpoints, True)

In [0]:
%sql
-- make silver cleaned table CDF enabled

-- ALTER TABLE retails.silver.order_items_cleaned
-- SET TBLPROPERTIES (
--     delta.enableChangeDataFeed = true
-- )

-- DESCRIBE TABLE EXTENDED retails.silver.order_items_cleaned;